<a href="https://colab.research.google.com/github/CalculatedContent/xgboost2ww/blob/main/notebooks/SpamBase_Hyperparameter_Sweep_Alpha_Trend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SpamBase Hyperparameter Sweep for Alpha Trends (Baseline-Centered Controlled Design)

This notebook replaces a broad mixed hyperparameter cloud with **local, controlled sweeps around one strong baseline model**.

Core objective: create model families where the effective XGBoost matrix changes systematically so held-out accuracy is more likely to show interpretable trends versus WeightWatcher alpha.

## Experimental Rules for Alpha-Trend Sweeps

1. Use one strong baseline model selected by **validation** performance.
2. Build local sweeps around baseline and change **one structural mechanism at a time**.
3. Keep train/validation/test split fixed for all runs.
4. Keep preprocessing and evaluation pipeline fixed.
5. Keep WeightWatcher analysis options fixed across all models.
6. Use repeated seeds (`BASELINE_SEEDS`) for each configuration and aggregate mean/std.
7. Use **W7 as the primary matrix** for cross-model comparability.
8. Optionally include W8 only as a secondary section when runtime permits.


## Google Drive checkpoint + local Mac compatibility

This cell keeps a Colab Google Drive checkpoint path and a local fallback path so the notebook runs both in Colab and locally on macOS/Linux.

In [ ]:

from pathlib import Path
import os

IN_COLAB = False
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

CHECKPOINT_ROOT_COLAB = Path('/content/drive/MyDrive/xgboost2ww_checkpoints')
CHECKPOINT_ROOT_LOCAL = Path('./checkpoints')
RESULTS_ROOT_LOCAL = Path('./results')
EXPERIMENT_NAME = 'spambase_alpha_trend_sweep'

if IN_COLAB:
    drive.mount('/content/drive')
    CHECKPOINT_ROOT = CHECKPOINT_ROOT_COLAB
else:
    CHECKPOINT_ROOT = CHECKPOINT_ROOT_LOCAL

CHECKPOINT_DIR = CHECKPOINT_ROOT / EXPERIMENT_NAME
RESULTS_DIR = RESULTS_ROOT_LOCAL / EXPERIMENT_NAME
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('IN_COLAB:', IN_COLAB)
print('CHECKPOINT_DIR:', CHECKPOINT_DIR)
print('RESULTS_DIR:', RESULTS_DIR)


In [ ]:
# Optional install cell (for Colab)
# !pip -q install xgboost weightwatcher xgboost2ww scikit-learn pandas matplotlib seaborn scipy


## Imports and global settings

In [ ]:

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

from scipy.stats import spearmanr, pearsonr

import xgboost as xgb
import weightwatcher as ww
from xgboost2ww import convert


In [ ]:

RNG = 123
np.random.seed(RNG)

TEST_SIZE = 0.20
VALID_SIZE_FROM_TRAIN = 0.20
BASELINE_SEEDS = [11, 29, 47]

W_MATRIX_PRIMARY = 'W7'
W_MATRIX_SECONDARY = 'W8'  # optional
RUN_SECONDARY_W8 = False   # set True if runtime budget allows

WW_ANALYZE_OPTIONS = dict(randomize=True, detX=True)
WW_MIN_TAIL = 10

BASELINE_NAME = 'spambase_hist_baseline_v1'

BASE_OBJECTIVE_PARAMS = {
    'objective': 'binary:logistic',
    'eval_metric': ['logloss', 'auc'],
    'tree_method': 'hist'
}


## Load SpamBase with deterministic fallback and fixed preprocessing

In [ ]:

SPAMBASE_FEATURES = [
    'word_freq_make', 'word_freq_address', 'word_freq_all', 'word_freq_3d', 'word_freq_our', 'word_freq_over',
    'word_freq_remove', 'word_freq_internet', 'word_freq_order', 'word_freq_mail', 'word_freq_receive', 'word_freq_will',
    'word_freq_people', 'word_freq_report', 'word_freq_addresses', 'word_freq_free', 'word_freq_business', 'word_freq_email',
    'word_freq_you', 'word_freq_credit', 'word_freq_your', 'word_freq_font', 'word_freq_000', 'word_freq_money',
    'word_freq_hp', 'word_freq_hpl', 'word_freq_george', 'word_freq_650', 'word_freq_lab', 'word_freq_labs',
    'word_freq_telnet', 'word_freq_857', 'word_freq_data', 'word_freq_415', 'word_freq_85', 'word_freq_technology',
    'word_freq_1999', 'word_freq_parts', 'word_freq_pm', 'word_freq_direct', 'word_freq_cs', 'word_freq_meeting',
    'word_freq_original', 'word_freq_project', 'word_freq_re', 'word_freq_edu', 'word_freq_table', 'word_freq_conference',
    'char_freq_;', 'char_freq_(', 'char_freq_[', 'char_freq_!', 'char_freq_$', 'char_freq_#',
    'capital_run_length_average', 'capital_run_length_longest', 'capital_run_length_total'
]


def load_spambase_data():
    try:
        ds = fetch_openml(name='spambase', version=1, as_frame=False, parser='auto')
        X = ds.data.astype(np.float32)
        y = np.asarray(ds.target)
        if y.dtype.kind in 'OUS':
            y = np.where(np.isin(y.astype(str), ['1', 'spam', 'yes', 'True']), 1, 0)
        y = y.astype(int)
        source = 'openml'
    except Exception as e:
        print(f'OpenML fetch failed ({type(e).__name__}: {e}); using UCI fallback CSV.')
        df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.data', header=None)
        X = df.iloc[:, :-1].to_numpy(dtype=np.float32)
        y = df.iloc[:, -1].to_numpy(dtype=int)
        source = 'uci_csv'

    if np.isnan(X).any():
        med = np.nanmedian(X, axis=0)
        inds = np.where(np.isnan(X))
        X[inds] = med[inds[1]]

    return X, y, source

X, y, data_source = load_spambase_data()

idx_train_full, idx_test = train_test_split(
    np.arange(len(y)), test_size=TEST_SIZE, random_state=RNG, stratify=y
)
idx_train, idx_valid = train_test_split(
    idx_train_full, test_size=VALID_SIZE_FROM_TRAIN, random_state=RNG, stratify=y[idx_train_full]
)

X_train, y_train = X[idx_train], y[idx_train]
X_valid, y_valid = X[idx_valid], y[idx_valid]
X_test, y_test = X[idx_test], y[idx_test]
X_train_valid, y_train_valid = X[idx_train_full], y[idx_train_full]

print('Data source:', data_source)
print('Shapes train/valid/test:', X_train.shape, X_valid.shape, X_test.shape)
print('Class balance train/valid/test:', y_train.mean(), y_valid.mean(), y_test.mean())


## Baseline anchor model

The baseline is the anchor point for all sweeps: it is selected by validation performance and every sweep perturbs only one mechanism from this fixed reference.

In [ ]:

BASELINE_CANDIDATES = [
    dict(max_depth=5, eta=0.08, n_estimators=1600, min_child_weight=3, subsample=0.85, colsample_bytree=0.85, gamma=0.0, reg_alpha=0.0, reg_lambda=1.0),
    dict(max_depth=6, eta=0.06, n_estimators=2200, min_child_weight=3, subsample=0.90, colsample_bytree=0.90, gamma=0.0, reg_alpha=0.0, reg_lambda=1.0),
    dict(max_depth=4, eta=0.10, n_estimators=1200, min_child_weight=5, subsample=0.80, colsample_bytree=0.85, gamma=0.0, reg_alpha=0.0, reg_lambda=2.0),
]


def train_single_config(params, seed):
    xgb_params = {**BASE_OBJECTIVE_PARAMS,
                  'seed': int(seed),
                  'max_depth': int(params['max_depth']),
                  'eta': float(params['eta']),
                  'subsample': float(params['subsample']),
                  'colsample_bytree': float(params['colsample_bytree']),
                  'min_child_weight': float(params['min_child_weight']),
                  'gamma': float(params['gamma']),
                  'reg_alpha': float(params['reg_alpha']),
                  'reg_lambda': float(params['reg_lambda'])}

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)
    dtrain_valid = xgb.DMatrix(X_train_valid, label=y_train_valid)
    dtest = xgb.DMatrix(X_test, label=y_test)

    model = xgb.train(
        params=xgb_params,
        dtrain=dtrain,
        num_boost_round=int(params['n_estimators']),
        evals=[(dtrain, 'train'), (dvalid, 'valid')],
        early_stopping_rounds=100,
        verbose_eval=False
    )

    best_round = int(getattr(model, 'best_iteration', int(params['n_estimators']) - 1) + 1)

    p_train = model.predict(dtrain, iteration_range=(0, best_round))
    p_valid = model.predict(dvalid, iteration_range=(0, best_round))
    p_test = model.predict(dtest, iteration_range=(0, best_round))

    metrics = dict(
        train_accuracy=accuracy_score(y_train, (p_train >= 0.5).astype(int)),
        valid_accuracy=accuracy_score(y_valid, (p_valid >= 0.5).astype(int)),
        test_accuracy=accuracy_score(y_test, (p_test >= 0.5).astype(int)),
        train_auc=roc_auc_score(y_train, p_train),
        valid_auc=roc_auc_score(y_valid, p_valid),
        test_auc=roc_auc_score(y_test, p_test),
        train_logloss=log_loss(y_train, p_train),
        valid_logloss=log_loss(y_valid, p_valid),
        test_logloss=log_loss(y_test, p_test),
        best_round=best_round,
    )

    return model, metrics, xgb_params


def build_baseline_params():
    rows = []
    for i, cand in enumerate(BASELINE_CANDIDATES):
        valid_scores = []
        for seed in BASELINE_SEEDS:
            _, m, _ = train_single_config(cand, seed)
            valid_scores.append(m['valid_accuracy'])
        rows.append({'candidate_id': i, 'params': cand, 'valid_mean': float(np.mean(valid_scores)), 'valid_std': float(np.std(valid_scores))})

    baseline_df = pd.DataFrame(rows).sort_values(['valid_mean', 'valid_std'], ascending=[False, True]).reset_index(drop=True)
    return baseline_df.iloc[0]['params'], baseline_df

BASELINE_PARAMS, baseline_selection_df = build_baseline_params()
print('BASELINE_NAME:', BASELINE_NAME)
print('BASELINE_SEEDS:', BASELINE_SEEDS)
print('BASELINE_PARAMS:', BASELINE_PARAMS)
baseline_selection_df


## Controlled local sweep grids

Each sweep changes one structural mechanism with all other parameters fixed at `BASELINE_PARAMS`.

In [ ]:

def _unique_sorted(values, round_to=6):
    vals = sorted(set([round(float(v), round_to) for v in values]))
    return vals


def build_local_sweeps(baseline_params):
    b = baseline_params

    n_base = int(b['n_estimators'])
    n_grid = sorted(set(max(50, int(round(n_base * s))) for s in [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0]))

    d_base = int(b['max_depth'])
    depth_grid = sorted(set(int(np.clip(d_base + d, 2, 12)) for d in [-2, -1, 0, 1, 2]))

    mcw_base = float(b['min_child_weight'])
    mcw_grid = sorted(set(max(1.0, round(mcw_base * s, 3)) for s in [0.25, 0.5, 1.0, 2.0, 4.0]))

    s_base = float(b['subsample'])
    subsample_grid = _unique_sorted([
        max(0.4, s_base - 0.30),
        max(0.5, s_base - 0.15),
        s_base,
        min(1.0, s_base + 0.10),
        1.0,
    ], round_to=3)

    eta_base = float(b['eta'])
    eta_grid = sorted(set(float(np.clip(eta_base * s, 0.01, 0.3)) for s in [0.5, 1.0, 2.0]))
    eta_n_pairs = []
    constant_budget = eta_base * n_base
    for eta in eta_grid:
        n = int(max(100, round(constant_budget / eta)))
        eta_n_pairs.append((eta, n))

    return {
        'n_estimators': n_grid,
        'max_depth': depth_grid,
        'min_child_weight': mcw_grid,
        'subsample': subsample_grid,
        'eta_x_n_estimators': eta_n_pairs,
    }

LOCAL_SWEEPS = build_local_sweeps(BASELINE_PARAMS)
LOCAL_SWEEPS


## Training loop (fixed protocol)

Purpose: evaluate each configuration with the same split, same metric stack, and same repeated seeds before WeightWatcher extraction.

In [ ]:

def extract_weightwatcher_metrics(details_df, matrix_kind):
    out = {
        'matrix_kind': matrix_kind,
        'alpha_primary': np.nan,
        'alpha_mean': np.nan,
        'alpha_min': np.nan,
        'n_valid_layers': 0,
        'number_of_layers_analyzed': 0,
        'ww_fit_success': False,
        'effective_rank': np.nan,
        'rank': np.nan,
        'trace': np.nan,
        'frobenius_norm': np.nan,
        'n_rows': np.nan,
        'n_cols': np.nan,
    }

    if not isinstance(details_df, pd.DataFrame) or details_df.empty:
        return out

    out['number_of_layers_analyzed'] = int(len(details_df))

    alpha_col = 'alpha' if 'alpha' in details_df.columns else None
    if alpha_col is not None:
        alpha_vals = pd.to_numeric(details_df[alpha_col], errors='coerce')
        tail_col = None
        for c in ['num_evals', 'num_pl_spikes', 'M']:
            if c in details_df.columns:
                tail_col = c
                break
        if tail_col is not None:
            tail_vals = pd.to_numeric(details_df[tail_col], errors='coerce')
            valid_mask = alpha_vals.notna() & np.isfinite(alpha_vals) & (tail_vals.fillna(WW_MIN_TAIL) >= WW_MIN_TAIL)
        else:
            valid_mask = alpha_vals.notna() & np.isfinite(alpha_vals)

        valid_alphas = alpha_vals[valid_mask]
        out['n_valid_layers'] = int(valid_alphas.shape[0])
        if out['n_valid_layers'] > 0:
            out['alpha_primary'] = float(np.median(valid_alphas))
            out['alpha_mean'] = float(np.mean(valid_alphas))
            out['alpha_min'] = float(np.min(valid_alphas))
            out['ww_fit_success'] = True

    for c in ['effective_rank', 'erank']:
        if c in details_df.columns:
            out['effective_rank'] = float(pd.to_numeric(details_df[c], errors='coerce').dropna().mean())
            break
    for c in ['rank']:
        if c in details_df.columns:
            out['rank'] = float(pd.to_numeric(details_df[c], errors='coerce').dropna().mean())
            break
    for c in ['trace', 'matrix_trace']:
        if c in details_df.columns:
            out['trace'] = float(pd.to_numeric(details_df[c], errors='coerce').dropna().mean())
            break
    for c in ['fro', 'frobenius_norm', 'norm']:
        if c in details_df.columns:
            out['frobenius_norm'] = float(pd.to_numeric(details_df[c], errors='coerce').dropna().mean())
            break
    for c in ['N', 'n_rows']:
        if c in details_df.columns:
            out['n_rows'] = float(pd.to_numeric(details_df[c], errors='coerce').dropna().mean())
            break
    for c in ['M', 'n_cols']:
        if c in details_df.columns:
            out['n_cols'] = float(pd.to_numeric(details_df[c], errors='coerce').dropna().mean())
            break

    return out


def run_config_across_seeds(sweep_name, sweep_value, params):
    run_rows, layer_rows = [], []
    for seed in BASELINE_SEEDS:
        model, metrics, train_params = train_single_config(params, seed)

        converted = convert(
            model=model,
            data=X_train_valid,
            labels=y_train_valid,
            W=W_MATRIX_PRIMARY,
            nfolds=5,
            t_points=40,
            random_state=RNG,
            train_params=train_params,
            num_boost_round=metrics['best_round'],
            multiclass='error',
            return_type='torch',
            verbose=False,
        )
        details = ww.WeightWatcher(model=converted).analyze(**WW_ANALYZE_OPTIONS)
        ww_metrics = extract_weightwatcher_metrics(details, W_MATRIX_PRIMARY)

        config_id = f"{sweep_name}::{sweep_value}"
        row = {
            'sweep_name': sweep_name,
            'sweep_value': sweep_value,
            'seed': seed,
            'configuration_id': config_id,
            'is_baseline': bool(sweep_name == 'baseline'),
            **params,
            **metrics,
            **ww_metrics,
        }
        row['train_test_gap'] = row['train_accuracy'] - row['test_accuracy']
        run_rows.append(row)

        if isinstance(details, pd.DataFrame) and not details.empty:
            d = details.copy()
            d['sweep_name'] = sweep_name
            d['sweep_value'] = sweep_value
            d['seed'] = seed
            d['configuration_id'] = config_id
            layer_rows.append(d)

    return run_rows, layer_rows


def aggregate_results(per_seed_df):
    metric_cols = ['train_accuracy', 'valid_accuracy', 'test_accuracy', 'alpha_primary', 'alpha_mean', 'alpha_min', 'effective_rank', 'train_test_gap']
    agg_dict = {c: ['mean', 'std'] for c in metric_cols}
    group_cols = ['sweep_name', 'sweep_value', 'configuration_id', 'is_baseline']
    agg_df = per_seed_df.groupby(group_cols, as_index=False).agg(agg_dict)
    agg_df.columns = ['_'.join(c).strip('_') for c in agg_df.columns]
    return agg_df


In [ ]:

def build_params_from_baseline(base, key, value):
    p = dict(base)
    p[key] = value
    return p

experiment_plan = []

# Baseline row
experiment_plan.append(dict(sweep_name='baseline', sweep_value='baseline', params=dict(BASELINE_PARAMS)))

# Primary sweeps
for v in LOCAL_SWEEPS['n_estimators']:
    experiment_plan.append(dict(sweep_name='n_estimators', sweep_value=v, params=build_params_from_baseline(BASELINE_PARAMS, 'n_estimators', int(v))))
for v in LOCAL_SWEEPS['max_depth']:
    experiment_plan.append(dict(sweep_name='max_depth', sweep_value=v, params=build_params_from_baseline(BASELINE_PARAMS, 'max_depth', int(v))))
for v in LOCAL_SWEEPS['min_child_weight']:
    experiment_plan.append(dict(sweep_name='min_child_weight', sweep_value=v, params=build_params_from_baseline(BASELINE_PARAMS, 'min_child_weight', float(v))))
for v in LOCAL_SWEEPS['subsample']:
    experiment_plan.append(dict(sweep_name='subsample', sweep_value=v, params=build_params_from_baseline(BASELINE_PARAMS, 'subsample', float(v))))

# Optional structured two-parameter sweep
for eta, n_estimators in LOCAL_SWEEPS['eta_x_n_estimators']:
    p = dict(BASELINE_PARAMS)
    p['eta'] = float(eta)
    p['n_estimators'] = int(n_estimators)
    experiment_plan.append(dict(sweep_name='eta_x_n_estimators', sweep_value=f'eta={eta:.4f}|n={n_estimators}', params=p))

len(experiment_plan)


## WeightWatcher extraction

Purpose: run one fixed WW pipeline (`W7`, same analyze options) for every trained model and extract standardized alpha and matrix metadata.

In [ ]:

all_rows = []
all_layer_rows = []

for item in experiment_plan:
    if (item['sweep_name'] == 'eta_x_n_estimators') and (not RUN_SECONDARY_W8):
        # still run this sweep because it is part of the design; W8 remains optional
        pass
    rows, layer_rows = run_config_across_seeds(item['sweep_name'], item['sweep_value'], item['params'])
    all_rows.extend(rows)
    all_layer_rows.extend(layer_rows)

per_seed_results_df = pd.DataFrame(all_rows)
layer_long_df = pd.concat(all_layer_rows, ignore_index=True) if all_layer_rows else pd.DataFrame()
aggregated_results_df = aggregate_results(per_seed_results_df)

print('Per-seed rows:', per_seed_results_df.shape)
print('Aggregated rows:', aggregated_results_df.shape)
print('Layer-long rows:', layer_long_df.shape)
per_seed_results_df.head()


## Plotting and trend diagnostics

Purpose: visualize whether each local sweep forms a coherent alpha ↔ performance trajectory instead of a heterogeneous cloud.

In [ ]:

def _get_baseline_agg_row(agg_df):
    return agg_df[agg_df['is_baseline'] == True].iloc[0]


def plot_sweep_results(agg_df, sweep_name):
    sdf = agg_df[agg_df['sweep_name'] == sweep_name].copy()
    if sdf.empty:
        print(f'No rows for sweep={sweep_name}')
        return

    baseline_row = _get_baseline_agg_row(agg_df)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    x = sdf['sweep_value'].astype(str)
    axes[0].plot(x, sdf['train_accuracy_mean'], marker='o', label='train')
    axes[0].plot(x, sdf['valid_accuracy_mean'], marker='o', label='valid')
    axes[0].plot(x, sdf['test_accuracy_mean'], marker='o', label='test')
    axes[0].set_title(f'{sweep_name}: accuracy vs sweep axis')
    axes[0].set_xlabel('sweep value')
    axes[0].set_ylabel('accuracy')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].legend()

    axes[1].plot(x, sdf['alpha_primary_mean'], marker='o', color='purple')
    axes[1].set_title(f'{sweep_name}: alpha_primary vs sweep axis')
    axes[1].set_xlabel('sweep value')
    axes[1].set_ylabel('alpha_primary')
    axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sc = axes[0].scatter(sdf['alpha_primary_mean'], sdf['test_accuracy_mean'], c=range(len(sdf)), cmap='viridis', s=90)
    for _, r in sdf.iterrows():
        axes[0].annotate(str(r['sweep_value']), (r['alpha_primary_mean'], r['test_accuracy_mean']), fontsize=8)
    axes[0].scatter([baseline_row['alpha_primary_mean']], [baseline_row['test_accuracy_mean']], marker='*', s=250, color='red', label='baseline')
    axes[0].set_xlabel('alpha_primary (mean)')
    axes[0].set_ylabel('test_accuracy (mean)')
    axes[0].set_title(f'{sweep_name}: test accuracy vs alpha_primary')
    axes[0].legend()

    axes[1].scatter(sdf['alpha_primary_mean'], sdf['train_test_gap_mean'], c='teal', s=90)
    for _, r in sdf.iterrows():
        axes[1].annotate(str(r['sweep_value']), (r['alpha_primary_mean'], r['train_test_gap_mean']), fontsize=8)
    axes[1].scatter([baseline_row['alpha_primary_mean']], [baseline_row['train_test_gap_mean']], marker='*', s=250, color='red', label='baseline')
    axes[1].set_xlabel('alpha_primary (mean)')
    axes[1].set_ylabel('train-test gap (mean)')
    axes[1].set_title(f'{sweep_name}: train-test gap vs alpha_primary')
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def plot_alpha_vs_accuracy(agg_df):
    plt.figure(figsize=(7, 5))
    sns.scatterplot(data=agg_df, x='alpha_primary_mean', y='test_accuracy_mean', hue='sweep_name', s=90)
    plt.title('All sweeps: test accuracy vs alpha_primary')
    plt.show()


In [ ]:

for sweep in ['n_estimators', 'max_depth', 'min_child_weight', 'subsample', 'eta_x_n_estimators']:
    print('
' + '=' * 30)
    print('Sweep:', sweep)
    plot_sweep_results(aggregated_results_df, sweep)

plot_alpha_vs_accuracy(aggregated_results_df)


### Sweep interpretation template

After each sweep plot, add brief notes on whether the trajectory appears smooth (trend-like) or cloud-like (noisy/non-monotonic). Use this section to document trend quality.

## Final summary: sweep quality and baseline/best-model comparison

In [ ]:

def safe_corr(x, y, fn):
    x = pd.to_numeric(x, errors='coerce')
    y = pd.to_numeric(y, errors='coerce')
    mask = x.notna() & y.notna()
    if mask.sum() < 2:
        return np.nan
    return float(fn(x[mask], y[mask])[0])

summary_rows = []
for sweep, sdf in aggregated_results_df.groupby('sweep_name'):
    baseline_value = 'baseline'
    if sweep in LOCAL_SWEEPS:
        baseline_key = BASELINE_PARAMS['n_estimators'] if sweep == 'n_estimators' else BASELINE_PARAMS[sweep]
        baseline_value = baseline_key

    summary_rows.append(dict(
        sweep_name=sweep,
        number_of_configs=int(len(sdf)),
        baseline_value=baseline_value,
        test_acc_range=float(sdf['test_accuracy_mean'].max() - sdf['test_accuracy_mean'].min()),
        alpha_range=float(sdf['alpha_primary_mean'].max() - sdf['alpha_primary_mean'].min()),
        spearman_alpha_test=safe_corr(sdf['alpha_primary_mean'], sdf['test_accuracy_mean'], spearmanr),
        pearson_alpha_test=safe_corr(sdf['alpha_primary_mean'], sdf['test_accuracy_mean'], pearsonr),
        spearman_alpha_gap=safe_corr(sdf['alpha_primary_mean'], sdf['train_test_gap_mean'], spearmanr),
        best_validation_configuration=sdf.sort_values('valid_accuracy_mean', ascending=False).iloc[0]['configuration_id'],
        best_test_configuration=sdf.sort_values('test_accuracy_mean', ascending=False).iloc[0]['configuration_id'],
    ))

sweep_summary_df = pd.DataFrame(summary_rows).sort_values('spearman_alpha_test', ascending=False)

best_validation_row = aggregated_results_df.sort_values('valid_accuracy_mean', ascending=False).iloc[0]
baseline_row = aggregated_results_df[aggregated_results_df['is_baseline'] == True].iloc[0]

comparison_df = pd.DataFrame([
    dict(model_label='baseline', configuration_id=baseline_row['configuration_id'], valid_accuracy=baseline_row['valid_accuracy_mean'], test_accuracy=baseline_row['test_accuracy_mean'], alpha_primary=baseline_row['alpha_primary_mean']),
    dict(model_label='best_validation_selected', configuration_id=best_validation_row['configuration_id'], valid_accuracy=best_validation_row['valid_accuracy_mean'], test_accuracy=best_validation_row['test_accuracy_mean'], alpha_primary=best_validation_row['alpha_primary_mean'])
])

print('Sweep summary (trend quality):')
display(sweep_summary_df)
print('Baseline vs strongest validation-selected model:')
display(comparison_df)


## Save artifacts (per-seed, aggregated, layer-long, summary)

These outputs are written to both local `results/` and checkpoint directories for easy reruns and recovery.

In [ ]:

per_seed_path_local = RESULTS_DIR / 'spambase_alpha_trend_per_seed.csv'
agg_path_local = RESULTS_DIR / 'spambase_alpha_trend_aggregated.csv'
layer_path_local = RESULTS_DIR / 'spambase_alpha_trend_layer_long.csv'
summary_path_local = RESULTS_DIR / 'spambase_alpha_trend_sweep_summary.csv'
comparison_path_local = RESULTS_DIR / 'spambase_alpha_trend_baseline_vs_best.csv'

per_seed_path_ckpt = CHECKPOINT_DIR / per_seed_path_local.name
agg_path_ckpt = CHECKPOINT_DIR / agg_path_local.name
layer_path_ckpt = CHECKPOINT_DIR / layer_path_local.name
summary_path_ckpt = CHECKPOINT_DIR / summary_path_local.name
comparison_path_ckpt = CHECKPOINT_DIR / comparison_path_local.name

per_seed_results_df.to_csv(per_seed_path_local, index=False)
aggregated_results_df.to_csv(agg_path_local, index=False)
layer_long_df.to_csv(layer_path_local, index=False)
sweep_summary_df.to_csv(summary_path_local, index=False)
comparison_df.to_csv(comparison_path_local, index=False)

per_seed_results_df.to_csv(per_seed_path_ckpt, index=False)
aggregated_results_df.to_csv(agg_path_ckpt, index=False)
layer_long_df.to_csv(layer_path_ckpt, index=False)
sweep_summary_df.to_csv(summary_path_ckpt, index=False)
comparison_df.to_csv(comparison_path_ckpt, index=False)

print('Saved local:', per_seed_path_local, agg_path_local, layer_path_local, summary_path_local, comparison_path_local)
print('Saved checkpoint:', per_seed_path_ckpt, agg_path_ckpt, layer_path_ckpt, summary_path_ckpt, comparison_path_ckpt)


## Notes

- Primary matrix family is W7 for strict comparability.
- W8 is optional secondary analysis (`RUN_SECONDARY_W8`).
- Keep all non-swept hyperparameters fixed at baseline for trend interpretability.
